In [0]:
# %sh
# mkdir /dbfs/spark_lab
# wget -O /dbfs/spark_lab/2019.csv df_2019
# wget -O /dbfs/spark_lab/2020.csv df_2020
# wget -O /dbfs/spark_lab/2021.csv df_2021

In [0]:
import pandas as pd

#Create variables to implement schema during reading with pandas

column_names = [
    "SalesOrderNumber", "SalesOrderLineNumber", "OrderDate", 
    "CustomerName", "Email", "Item", "Quantity", "UnitPrice", "Tax"
]

dtype_dict = {
    "SalesOrderNumber": str,
    "SalesOrderLineNumber": int,
    "OrderDate": str, 
    "CustomerName": str,
    "Email": str,
    "Item": str,
    "Quantity": int,
    "UnitPrice": float,
    "Tax": float
}



In [0]:
#Reading every file individually, then concatenate them and convert into spark DF 

url = 'https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/2019.csv'

df_2019 = pd.read_csv(url , names=column_names, header=None, dtype=dtype_dict)

url = 'https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/2020.csv'

df_2020 = pd.read_csv(url, names=column_names, header=None, dtype=dtype_dict)

url = 'https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/2021.csv'

df_2021 = pd.read_csv(url, names=column_names, header=None, dtype=dtype_dict)

df_2019_2021 = pd.concat([df_2019,df_2020,df_2021],axis=0)

df_2019_2021["OrderDate"] = pd.to_datetime(df_2019_2021["OrderDate"], errors='coerce')

df = spark.createDataFrame(df_2019_2021)




In [0]:
# Create a temporary view NAMED "salesorders". 

df.createOrReplaceTempView("salesorders")
spark_df = spark.sql("SELECT * FROM salesorders")
display(spark_df)

In [0]:

%sql

--Here we-re using the %sql ('Magic') to force the cell to run as SQL 
    
SELECT * FROM salesorders

In [0]:
# This is other way to execute SQL in databricks, using the spark.sql() function of a Spark DF

sqlQuery = "SELECT CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear, \
                SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue \
         FROM salesorders \
         GROUP BY CAST(YEAR(OrderDate) AS CHAR(4)) \
         ORDER BY OrderYear"
df_spark = spark.sql(sqlQuery)
df_spark.show()

In [0]:
from matplotlib import pyplot as plt

#Revertirmos el tipo de DF de Spark a Pandas, para tener compatibilidad con matplotlib

# matplotlib requires a Pandas dataframe, not a Spark one
df_sales = df_spark.toPandas()
# Create a bar plot of revenue by year
plt.bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'])
# Display the plot
plt.show()

In [0]:
# Clear the plot area
plt.clf()
# Create a bar plot of revenue by year
plt.bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'], color='orange')
# Customize the chart
plt.title('Revenue by Year')
plt.xlabel('Year')
plt.ylabel('Revenue')
plt.grid(color='#95a5a6', linestyle='--', linewidth=2, axis='y', alpha=0.7)
plt.xticks(rotation=45)
# Show the figure
plt.show()

In [0]:
# Clear the plot area
plt.clf()
# Create a Figure
fig = plt.figure(figsize=(8,3))
# Create a bar plot of revenue by year
plt.bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'], color='orange')
# Customize the chart
plt.title('Revenue by Year')
plt.xlabel('Year')
plt.ylabel('Revenue')
plt.grid(color='#95a5a6', linestyle='--', linewidth=2, axis='y', alpha=0.7)
plt.xticks(rotation=45)
# Show the figure
plt.show()

In [0]:
# Clear the plot area
plt.clf()
# Create a figure for 2 subplots (1 row, 2 columns)
fig, ax = plt.subplots(1, 2, figsize = (10,4))
# Create a bar plot of revenue by year on the first axis
ax[0].bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'], color='orange')
ax[0].set_title('Revenue by Year')
# Create a pie chart of yearly order counts on the second axis
yearly_counts = df_sales['OrderYear'].value_counts()
ax[1].pie(yearly_counts)
ax[1].set_title('Orders per Year')
ax[1].legend(yearly_counts.keys().tolist())
# Add a title to the Figure
fig.suptitle('Sales Data')
# Show the figure
plt.show()

In [0]:
import seaborn as sns
   
# Clear the plot area
plt.clf()
# Create a bar chart
ax = sns.barplot(x="OrderYear", y="GrossRevenue", data=df_sales)
plt.show()

In [0]:
# Clear the plot area
plt.clf()
   
# Set the visual theme for seaborn
sns.set_theme(style="whitegrid")
   
# Create a bar chart
ax = sns.barplot(x="OrderYear", y="GrossRevenue", data=df_sales)
plt.show()

In [0]:
# Clear the plot area
plt.clf()
   
# Create a bar chart
ax = sns.lineplot(x="OrderYear", y="GrossRevenue", data=df_sales)
plt.show()